In [1]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "nested-b-d"   # the branch with the code this notebook needs (set back to "main" after merging)

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", url, CODE], check=True)
# fetch + checkout + pull, so a clone left on another branch earlier in the session switches over
subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", CODE, "pull", "-q", "origin", BRANCH], check=True)
print(BRANCH, subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
_MINE = {"fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic",
         "stage1_structural", "nested_builder"}
for _m in list(sys.modules):
    if _m in _MINE or _m == "src" or _m.startswith("src."):   # src.* = the team's nested code
        sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

Mounted at /content/drive
nested-b-d fed9369 nested variants



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ammara-asif125 (kashaf-saleem09-institute-of-business-administration) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs', 'checkpoints': '/content/drive/MyDrive/fyp_nested_cl/checkpoints'}
GPU: Tesla T4


In [2]:
import numpy as np, pandas as pd

def table(configs):
    """One row per config: accuracy metrics, cost, and per-level means over tasks (nested arms)."""
    rows = []
    for cfg in configs:
        base = {"arm": sr.arm_label(cfg), "lr": cfg["lr"], "seed": cfg["seed"]}
        r = load_result(cfg, paths)
        if r is None:
            rows.append({**base, "status": "not finished"}); continue
        m = r["metrics"]
        row = {**base, "run_id": r["run_id"], "avg_incremental_acc": m["avg_incremental_acc"],
               "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
               # mean of the diagonal: accuracy on each task right after learning it (plasticity check)
               "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
               "steps": m["optimizer_steps"], "n_params": m["n_params"],
               "minutes": r["wall_clock_s"] / 60, "peak_gpu_mb": m.get("peak_gpu_mem_mb")}
        log = m.get("level_log_per_exp") or []
        for k in (log[0] if log else {}):
            if k.startswith(("updates/", "rel_displacement/", "penalty_ratio/")):
                row[k] = float(np.mean([x[k] for x in log]))
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], cwd=CODE, capture_output=True, text=True,
                   env={**os.environ, "WANDB_MODE": "disabled", "CUDA_VISIBLE_DEVICES": ""})   # CPU: deterministic
print(r.stdout[-6000:]); print(r.stderr[-3000:])
assert r.returncode == 0, "tests failed: read the output above before running anything else"

..............                                                           [100%]
=============================== warnings summary ===============================
tests/test_nested.py::test_flat_nested_path_equals_naive
tests/test_nested.py::test_flat_nested_path_equals_naive
tests/test_nested.py::test_b_per_level_lrs
tests/test_nested.py::test_d_runs_steps_and_checkpoints
tests/test_smoke.py::test_end_to_end[extra0]
tests/test_smoke.py::test_end_to_end[extra1]
tests/test_smoke.py::test_end_to_end[extra2]
tests/test_smoke.py::test_llrd_matches_flat_params_and_steps
tests/test_smoke.py::test_llrd_matches_flat_params_and_steps
  /usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
    warnings.warn("No loggers specified, metrics will not be logged")

tests/test_nested.py::test_flat_nested_path_equals_naive
tests/test_nested.py::test_b_per_level_lrs
tests/test_nested.py::test_d_runs_steps_and_check

In [ ]:
SMOKE = [sr.arm_cfg(name, stage="smoke_nested", n_experiences=2, epochs=1, lr=0.1, eval_split="val")
         for name in ["flat", "nested_b", "nested_d"]]
out = run_sweep(SMOKE, sr.run_experiment, paths, force=True)   # force: always re-time after code changes
smoke = table(SMOKE)
smoke["train_min"] = [sum(out[run_id(c)]["metrics"]["train_minutes_per_exp"]) if out[run_id(c)] else None
                      for c in SMOKE]
smoke["est_full_run_train_min"] = 10 * smoke["train_min"]
display(smoke.T)

[1/3] 2bc669a58d2a running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/2: seen-avg 18.28%  current 18.28%  train-loss 3.354
  exp 2/2: seen-avg 9.92%  current 19.84%  train-loss 3.418


acc_current_exp,▁█
acc_exp/00,█▁
acc_exp/01,▁
acc_seen_avg,█▁
exp,▁█
level/displacement/layer2+layer3,█▁
level/displacement/layer4+fc,▁█
level/displacement/stem+layer1,█▁
level/rel_displacement/layer2+layer3,█▁
level/rel_displacement/layer4+fc,▁█
+7,...


[2/3] 1c1ba510bd53 running


...

[3/3] ffbb0bb80f16 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/2: seen-avg 12.52%  current 12.52%  train-loss 3.783
  exp 2/2: seen-avg 6.98%  current 13.96%  train-loss 3.967


acc_current_exp,▁█
acc_exp/00,█▁
acc_exp/01,▁
acc_seen_avg,█▁
exp,▁█
level/data_grad_norm/layer2+layer3,█▁
level/data_grad_norm/layer4+fc,▁█
level/data_grad_norm/stem+layer1,█▁
level/displacement/layer2+layer3,█▁
level/displacement/layer4+fc,▁█
+16,...


,0,1,2
arm,flat_nl,nested_b,nested_d_lam0.1
lr,0.1,0.1,0.1
seed,0,0,0
run_id,2bc669a58d2a,1c1ba510bd53,ffbb0bb80f16
avg_incremental_acc,14.1,14.15,9.75
final_avg_acc,9.92,9.7,6.98
avg_forgetting,18.28,18.6,12.52
acc_just_learned,19.06,19.0,13.24
steps,1408,1408,1408
n_params,1109240,1109240,1109240


**Variant B:**

In [3]:
EPOCHS = 10

def val_cfg(name, lr, **kw):
    return sr.arm_cfg(name, stage="stage1_nested_lr_grid", epochs=EPOCHS, seed=0, lr=lr, eval_split="val", **kw)

OLD_B = {lr: sr.make_cfg(stage="stage1_lr_grid", strategy="naive", epochs=EPOCHS, seed=0, lr=lr, eval_split="val",
                         lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="fast_deep")
         for lr in [0.03, 0.1, 0.2, 0.3, 0.5]}
B_GRID = [0.5, 0.6, 0.7, 0.9]     # 0.5 first: it is the regression run
B_CONFIGS = [val_cfg("nested_b", lr) for lr in B_GRID]
for c in B_CONFIGS: print(run_id(c), sr.arm_label(c), c["lr"])

50adb94f6284 nested_b 0.5
5545124984d8 nested_b 0.6
b66e25fa8b0b nested_b 0.7
ee4c2b8d3c46 nested_b 0.9


In [ ]:
out = run_sweep(B_CONFIGS, sr.run_experiment, paths)

[1/4] 50adb94f6284 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 73.20%  current 73.20%  train-loss 0.903
  exp 2/20: seen-avg 44.40%  current 88.80%  train-loss 0.498
  exp 3/20: seen-avg 24.67%  current 74.00%  train-loss 0.840
  exp 4/20: seen-avg 19.80%  current 79.20%  train-loss 0.505
  exp 5/20: seen-avg 14.88%  current 74.40%  train-loss 0.836
  exp 6/20: seen-avg 12.93%  current 77.60%  train-loss 0.720
  exp 7/20: seen-avg 12.57%  current 88.00%  train-loss 0.590
  exp 8/20: seen-avg 10.80%  current 86.40%  train-loss 0.448
  exp 9/20: seen-avg 9.33%  current 84.00%  train-loss 0.577
  exp 10/20: seen-avg 8.60%  current 86.00%  train-loss 0.544
  exp 11/20: seen-avg 7.42%  current 81.60%  train-loss 0.533
  exp 12/20: seen-avg 6.30%  current 75.60%  train-loss 0.673
  exp 13/20: seen-avg 5.66%  current 73.60%  train-loss 0.731
  exp 14/20: seen-avg 6.37%  current 89.20%  train-loss 0.458
  exp 15/20: seen-avg 5.76%  current 86.40%  train-loss 0.617
  exp 16/20: seen-avg 5.12%  current 82.00%  train-loss 0.499
  exp 17/

acc_current_exp,▁█▁▄▂▃▇▇▆▇▅▂▁█▇▅▇▇▅▇
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


[2/4] 5545124984d8 running


...

acc_current_exp,▃█▃▆▄▁▆▆▅▆▄▂▅▇▆█▇▇▆█
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


[4/4] ee4c2b8d3c46 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 70.00%  current 70.00%  train-loss 0.959
  exp 2/20: seen-avg 44.20%  current 88.40%  train-loss 0.539
  exp 3/20: seen-avg 26.80%  current 80.40%  train-loss 0.814
  exp 4/20: seen-avg 21.90%  current 87.60%  train-loss 0.491
  exp 5/20: seen-avg 15.12%  current 75.60%  train-loss 0.783
  exp 6/20: seen-avg 12.13%  current 72.80%  train-loss 0.714
  exp 7/20: seen-avg 10.91%  current 76.40%  train-loss 0.614
  exp 8/20: seen-avg 10.75%  current 86.00%  train-loss 0.448
  exp 9/20: seen-avg 8.98%  current 80.80%  train-loss 0.580
  exp 10/20: seen-avg 8.12%  current 81.20%  train-loss 0.533
  exp 11/20: seen-avg 7.27%  current 80.00%  train-loss 0.542
  exp 12/20: seen-avg 6.17%  current 74.00%  train-loss 0.649
  exp 13/20: seen-avg 6.25%  current 81.20%  train-loss 0.746
  exp 14/20: seen-avg 6.20%  current 86.80%  train-loss 0.443
  exp 15/20: seen-avg 5.71%  current 85.60%  train-loss 0.592
  exp 16/20: seen-avg 5.40%  current 86.40%  train-loss 0.475
  exp 17/

acc_current_exp,▁█▅█▃▂▃▇▅▅▅▂▅▇▇▇██▅▇
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


In [ ]:
old, new = load_result(OLD_B[0.5], paths), load_result(B_CONFIGS[0], paths)
assert old is not None, "old llrd_fast_deep lr 0.5 val run not found on Drive"
if new is not None:
    reg = table([OLD_B[0.5], B_CONFIGS[0]])
    display(reg[["arm", "lr", "run_id", "avg_incremental_acc", "acc_just_learned", "final_avg_acc", "avg_forgetting", "minutes"]])
    d_inc = abs(reg["avg_incremental_acc"].iloc[0] - reg["avg_incremental_acc"].iloc[1])
    d_jl = abs(reg["acc_just_learned"].iloc[0] - reg["acc_just_learned"].iloc[1])
    A_old, A_new = np.array(old["metrics"]["acc_matrix"]), np.array(new["metrics"]["acc_matrix"])
    print(f"|diff| avg incremental {d_inc:.2f}, just-learned {d_jl:.2f}; acc-matrix max |diff| {np.abs(A_old - A_new).max():.2f}")
    print("REGRESSION CHECK", "PASSED" if max(d_inc, d_jl) < 0.5 else "FAILED: stop and investigate")

,arm,lr,run_id,avg_incremental_acc,acc_just_learned,final_avg_acc,avg_forgetting,minutes
0,llrd_fast_deep,0.5,324e0679f865,14.333412,81.2,4.28,80.968421,8.450855
1,nested_b,0.5,50adb94f6284,14.324887,82.2,4.36,81.936842,7.474843


|diff| avg incremental 0.01, just-learned 1.00; acc-matrix max |diff| 8.80
REGRESSION CHECK FAILED: stop and investigate


In [ ]:
bdf = table(list(OLD_B.values()) + B_CONFIGS)
display(bdf[["arm", "lr", "avg_incremental_acc", "acc_just_learned", "final_avg_acc", "minutes"]])
done = bdf[bdf["arm"] == "nested_b"].dropna(subset=["avg_incremental_acc"])
if len(done):
    best = done.loc[done["avg_incremental_acc"].idxmax()]
    print(f"best B lr on its grid: {best['lr']} (avg incremental {best['avg_incremental_acc']:.2f})")
    if best["lr"] == max(B_GRID): print("WARNING: best is at the top of the grid; extend B_GRID before choosing.")

,arm,lr,avg_incremental_acc,acc_just_learned,final_avg_acc,minutes
0,llrd_fast_deep,0.03,12.239251,69.06,3.70,8.952716
1,llrd_fast_deep,0.10,13.549951,76.50,4.02,8.460360
2,llrd_fast_deep,0.20,14.189952,80.04,4.26,8.434010
3,llrd_fast_deep,0.30,14.130836,80.74,4.32,8.408175
4,llrd_fast_deep,0.50,14.333412,81.20,4.28,8.450855
5,nested_b,0.50,14.324887,82.20,4.36,7.474843
6,nested_b,0.60,14.412134,82.14,4.34,7.336550
7,nested_b,0.70,14.383561,82.18,4.50,7.356404
8,nested_b,0.90,14.230633,81.88,4.34,7.400746


best B lr on its grid: 0.6 (avg incremental 14.41)


**Variant D**

In [6]:
D_LR_GRID = [0.75, 0.80]
D_LR_CONFIGS = [val_cfg("nested_d", lr) for lr in D_LR_GRID]   # surprise_weight 0.1 from nested_d.yaml
for c in D_LR_CONFIGS: print(run_id(c), sr.arm_label(c), c["lr"])
out = run_sweep(D_LR_CONFIGS, sr.run_experiment, paths)

ddf56df71e42 nested_d_lam0.1 0.75
584b95ec6670 nested_d_lam0.1 0.8
[1/2] ddf56df71e42 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 70.00%  current 70.00%  train-loss 1.099
  exp 2/20: seen-avg 43.40%  current 86.80%  train-loss 0.810
  exp 3/20: seen-avg 22.40%  current 67.20%  train-loss 1.238
  exp 4/20: seen-avg 21.20%  current 84.80%  train-loss 0.781
  exp 5/20: seen-avg 14.00%  current 70.00%  train-loss 1.320
  exp 6/20: seen-avg 11.67%  current 70.00%  train-loss 1.169
  exp 7/20: seen-avg 11.43%  current 80.00%  train-loss 0.945
  exp 8/20: seen-avg 10.40%  current 83.20%  train-loss 0.862
  exp 9/20: seen-avg 8.53%  current 76.80%  train-loss 0.956
  exp 10/20: seen-avg 7.84%  current 78.40%  train-loss 0.867
  exp 11/20: seen-avg 6.98%  current 76.80%  train-loss 0.776
  exp 12/20: seen-avg 5.93%  current 71.20%  train-loss 0.870
  exp 13/20: seen-avg 5.57%  current 72.40%  train-loss 1.107
  exp 14/20: seen-avg 6.17%  current 86.40%  train-loss 0.846
  exp 15/20: seen-avg 5.12%  current 76.80%  train-loss 0.840
  exp 16/20: seen-avg 5.22%  current 83.60%  train-loss 0.684
  exp 17/

acc_current_exp,▂█▁▇▂▂▆▇▄▅▄▂▃█▄▇█▇▄▆
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+34,...


[2/2] 584b95ec6670 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 67.20%  current 67.20%  train-loss 1.094
  exp 2/20: seen-avg 42.40%  current 84.80%  train-loss 0.752
  exp 3/20: seen-avg 19.73%  current 59.20%  train-loss 1.564
  exp 4/20: seen-avg 20.40%  current 81.60%  train-loss 1.309
  exp 5/20: seen-avg 12.80%  current 64.00%  train-loss 1.324
  exp 6/20: seen-avg 12.00%  current 72.00%  train-loss 1.229
  exp 7/20: seen-avg 11.37%  current 79.60%  train-loss 1.140
  exp 8/20: seen-avg 10.15%  current 81.20%  train-loss 0.907
  exp 9/20: seen-avg 8.22%  current 74.00%  train-loss 1.197
  exp 10/20: seen-avg 7.80%  current 78.00%  train-loss 1.144
  exp 11/20: seen-avg 6.76%  current 74.40%  train-loss 0.992
  exp 12/20: seen-avg 5.67%  current 68.00%  train-loss 1.483
  exp 13/20: seen-avg 5.38%  current 70.00%  train-loss 1.153
  exp 14/20: seen-avg 5.89%  current 82.40%  train-loss 0.852
  exp 15/20: seen-avg 5.25%  current 78.80%  train-loss 1.346
  exp 16/20: seen-avg 5.25%  current 84.00%  train-loss 1.029
  exp 17/

acc_current_exp,▃█▁▇▂▄▇▇▅▆▅▃▄▇▆██▇▅▆
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+34,...


In [7]:
ddf = table(D_LR_CONFIGS)
display(ddf.drop(columns=["seed", "run_id"], errors="ignore"))
done = ddf.dropna(subset=["avg_incremental_acc"]) if "avg_incremental_acc" in ddf else ddf.iloc[:0]
if len(done):
    best = done.loc[done["avg_incremental_acc"].idxmax()]
    print(f"best D lr: {best['lr']} (avg incremental {best['avg_incremental_acc']:.2f})")
    if best["lr"] in (min(D_LR_GRID), max(D_LR_GRID)): print("WARNING: best lr is at the edge of the grid; extend it first.")

,arm,lr,avg_incremental_acc,final_avg_acc,avg_forgetting,acc_just_learned,steps,n_params,minutes,peak_gpu_mb,updates/stem+layer1,rel_displacement/stem+layer1,updates/layer2+layer3,rel_displacement/layer2+layer3,updates/layer4+fc,rel_displacement/layer4+fc,penalty_ratio/stem+layer1,penalty_ratio/layer2+layer3
0,nested_d_lam0.1,0.75,13.676716,4.0,77.852632,77.96,14200,1109240,18.023481,207.792969,44.0,0.109591,177.0,0.079100,710.0,0.148634,5.420268,0.923474
1,nested_d_lam0.1,0.80,13.169660,3.9,75.578947,75.70,14200,1109240,17.972728,202.455566,44.0,0.120478,177.0,0.081738,710.0,0.160120,16.852419,0.830404


best D lr: 0.75 (avg incremental 13.68)


In [8]:
D_LR = 0.7                         # <- fill in from the table above
# Re-centred after the smoke test (lr 0.1, lambda 0.1): penalty/data gradient ratio was 3.9 (stem+layer1) and
# 1.2 (layer2+layer3), i.e. the penalty dominated. This grid spans ratio ~0.1 to ~4 on the slow level.
# 0.1 at D_LR is already in the lr grid (same config -> reused). 0 = variant A at the same lr.
LAM_GRID = [0.0, 0.003, 0.01, 0.03, 0.1]
if D_LR is None:
    print("set D_LR first")
else:
    D_LAM_CONFIGS = [val_cfg("nested_d", D_LR, rule_kwargs=dict(surprise_weight=lam)) for lam in LAM_GRID]
    for c in D_LAM_CONFIGS: print(run_id(c), sr.arm_label(c), c["lr"])
    out = run_sweep(D_LAM_CONFIGS, sr.run_experiment, paths)
    display(table(D_LAM_CONFIGS).drop(columns=["seed", "run_id"], errors="ignore"))

4946950b08e0 nested_d_lam0 0.7
69beb6811d79 nested_d_lam0.003 0.7
8c6d978ba839 nested_d_lam0.01 0.7
b7b84225ff81 nested_d_lam0.03 0.7
3b975fe194c5 nested_d_lam0.1 0.7
[1/5] 4946950b08e0 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 69.20%  current 69.20%  train-loss 0.980
  exp 2/20: seen-avg 42.80%  current 85.60%  train-loss 0.542
  exp 3/20: seen-avg 25.33%  current 76.00%  train-loss 0.892
  exp 4/20: seen-avg 21.40%  current 85.60%  train-loss 0.497
  exp 5/20: seen-avg 15.68%  current 78.40%  train-loss 0.806
  exp 6/20: seen-avg 13.07%  current 78.40%  train-loss 0.697
  exp 7/20: seen-avg 11.77%  current 82.40%  train-loss 0.596
  exp 8/20: seen-avg 10.55%  current 84.40%  train-loss 0.421
  exp 9/20: seen-avg 8.98%  current 80.80%  train-loss 0.566
  exp 10/20: seen-avg 8.92%  current 89.20%  train-loss 0.518
  exp 11/20: seen-avg 7.27%  current 80.00%  train-loss 0.555
  exp 12/20: seen-avg 6.23%  current 74.80%  train-loss 0.652
  exp 13/20: seen-avg 6.34%  current 82.40%  train-loss 0.727
  exp 14/20: seen-avg 6.11%  current 85.60%  train-loss 0.459
  exp 15/20: seen-avg 5.44%  current 81.60%  train-loss 0.619
  exp 16/20: seen-avg 5.50%  current 88.00%  train-loss 0.497
  exp 17/

acc_current_exp,▁▇▃▇▄▄▅▆▅█▅▃▅▇▅▇█▇▅█
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+34,...


[2/5] 69beb6811d79 running


...

/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 67.60%  current 67.60%  train-loss 1.006
  exp 2/20: seen-avg 44.00%  current 88.00%  train-loss 0.583
  exp 3/20: seen-avg 23.47%  current 70.40%  train-loss 0.937
  exp 4/20: seen-avg 22.40%  current 89.60%  train-loss 0.583
  exp 5/20: seen-avg 15.68%  current 78.40%  train-loss 0.858
  exp 6/20: seen-avg 13.33%  current 80.00%  train-loss 0.788
  exp 7/20: seen-avg 11.77%  current 82.40%  train-loss 0.647
  exp 8/20: seen-avg 11.00%  current 88.00%  train-loss 0.445
  exp 9/20: seen-avg 9.02%  current 81.20%  train-loss 0.592
  exp 10/20: seen-avg 8.56%  current 85.60%  train-loss 0.555
  exp 11/20: seen-avg 7.38%  current 81.20%  train-loss 0.560
  exp 12/20: seen-avg 6.40%  current 76.80%  train-loss 0.683
  exp 13/20: seen-avg 6.25%  current 81.20%  train-loss 0.783
  exp 14/20: seen-avg 5.89%  current 82.40%  train-loss 0.478
  exp 15/20: seen-avg 5.65%  current 84.80%  train-loss 0.632
  exp 16/20: seen-avg 5.30%  current 84.80%  train-loss 0.513
  exp 17/

acc_current_exp,▁▇▂█▄▅▆▇▅▇▅▄▅▆▆▆▇▇▆█
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+34,...


[5/5] 3b975fe194c5 already finished, skipping


,arm,lr,avg_incremental_acc,final_avg_acc,avg_forgetting,acc_just_learned,steps,n_params,minutes,peak_gpu_mb,updates/stem+layer1,rel_displacement/stem+layer1,updates/layer2+layer3,rel_displacement/layer2+layer3,updates/layer4+fc,rel_displacement/layer4+fc,penalty_ratio/stem+layer1,penalty_ratio/layer2+layer3
0,nested_d_lam0,0.7,14.179952,4.48,82.252632,82.62,14200,1109240,18.304550,207.792969,44.0,0.065322,177.0,0.096227,710.0,0.199419,0.000000,0.000000
1,nested_d_lam0.003,0.7,14.275520,4.40,83.031579,83.28,14200,1109240,18.240966,213.130371,44.0,0.066875,177.0,0.093973,710.0,0.192056,0.129019,0.020313
2,nested_d_lam0.01,0.7,14.303621,4.38,81.663158,81.96,14200,1109240,18.077507,202.455566,44.0,0.070722,177.0,0.090265,710.0,0.187572,0.418715,0.071843
3,nested_d_lam0.03,0.7,14.112607,4.42,81.894737,82.22,14200,1109240,17.751698,207.792969,44.0,0.083241,177.0,0.081616,710.0,0.171323,1.228223,0.237494
4,nested_d_lam0.1,0.7,13.733006,4.00,77.473684,77.60,14200,1109240,18.272273,202.455566,44.0,0.107483,177.0,0.071576,710.0,0.141667,10.540826,1.025621


In [9]:
SEEDS = [0, 1, 2]
LR = {"flat": 0.1, "nested_b": 0.6, "nested_d": 0.7}
D_LAM = 0.01
CONTROL = []
for s in SEEDS:                       # seed outer: each seed completes every arm before the next starts
    for name, lr in LR.items():
        if lr is None or (name == "nested_d" and D_LAM is None):
            continue
        kw = dict(rule_kwargs=dict(surprise_weight=D_LAM)) if name == "nested_d" else {}
        CONTROL.append(sr.arm_cfg(name, stage="stage1_nested_control", epochs=EPOCHS, seed=s, lr=lr,
                                  eval_split="test", save_checkpoints=True, **kw))
for name, lr in LR.items():
    if lr is None or (name == "nested_d" and D_LAM is None): print(f"skipping {name}: set LR / D_LAM first")
for c in CONTROL: print(run_id(c), sr.arm_label(c), c["lr"], c["seed"])

07a97dd12ec3 flat_nl 0.1 0
ea0983312db5 nested_b 0.6 0
b2316f2cb5f4 nested_d_lam0.01 0.7 0
78e133183087 flat_nl 0.1 1
48523f3dfeea nested_b 0.6 1
e3be0b8051fd nested_d_lam0.01 0.7 1
b09e3cc29bda flat_nl 0.1 2
badf7f571542 nested_b 0.6 2
c4b3004f3b61 nested_d_lam0.01 0.7 2


In [10]:
out = run_sweep(CONTROL, sr.run_experiment, paths)

[1/9] 07a97dd12ec3 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 68.80%  current 68.80%  train-loss 0.900
  exp 2/20: seen-avg 45.10%  current 90.20%  train-loss 0.552
  exp 3/20: seen-avg 23.93%  current 71.80%  train-loss 0.824
  exp 4/20: seen-avg 21.55%  current 86.20%  train-loss 0.563
  exp 5/20: seen-avg 16.24%  current 81.20%  train-loss 0.812
  exp 6/20: seen-avg 13.10%  current 78.60%  train-loss 0.760
  exp 7/20: seen-avg 11.97%  current 83.80%  train-loss 0.654
  exp 8/20: seen-avg 9.55%  current 76.40%  train-loss 0.502
  exp 9/20: seen-avg 8.49%  current 76.40%  train-loss 0.603
  exp 10/20: seen-avg 8.12%  current 81.20%  train-loss 0.586
  exp 11/20: seen-avg 7.40%  current 81.40%  train-loss 0.574
  exp 12/20: seen-avg 6.50%  current 78.00%  train-loss 0.707
  exp 13/20: seen-avg 6.08%  current 79.00%  train-loss 0.767
  exp 14/20: seen-avg 6.46%  current 90.40%  train-loss 0.513
  exp 15/20: seen-avg 5.44%  current 81.60%  train-loss 0.651
  exp 16/20: seen-avg 5.34%  current 85.40%  train-loss 0.543
  exp 17/2

acc_current_exp,▁█▂▇▅▄▆▃▃▅▅▄▄█▅▆██▅▅
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


[2/9] ea0983312db5 running


...

acc_current_exp,▃▇▆▂▆▄▆▇█▆▆▇▁▇▆▅▁▇▇▃
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


[9/9] c4b3004f3b61 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 74.60%  current 74.60%  train-loss 0.919
  exp 2/20: seen-avg 39.10%  current 78.20%  train-loss 0.720
  exp 3/20: seen-avg 27.67%  current 83.00%  train-loss 0.673
  exp 4/20: seen-avg 19.10%  current 76.40%  train-loss 0.780
  exp 5/20: seen-avg 17.12%  current 85.60%  train-loss 0.608
  exp 6/20: seen-avg 13.70%  current 82.20%  train-loss 0.658
  exp 7/20: seen-avg 11.80%  current 82.60%  train-loss 0.574
  exp 8/20: seen-avg 10.95%  current 87.60%  train-loss 0.583
  exp 9/20: seen-avg 10.33%  current 93.00%  train-loss 0.447
  exp 10/20: seen-avg 8.24%  current 82.40%  train-loss 0.743
  exp 11/20: seen-avg 7.82%  current 86.00%  train-loss 0.539
  exp 12/20: seen-avg 7.68%  current 92.20%  train-loss 0.397
  exp 13/20: seen-avg 5.45%  current 70.80%  train-loss 0.776
  exp 14/20: seen-avg 5.90%  current 82.60%  train-loss 0.496
  exp 15/20: seen-avg 5.79%  current 86.80%  train-loss 0.563
  exp 16/20: seen-avg 5.33%  current 85.20%  train-loss 0.527
  exp 17

acc_current_exp,▃▄▅▃▆▅▅▆█▅▆█▂▅▆▆▁▇▇▅
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+34,...


In [11]:
OLD_CONTROL = [sr.make_cfg(stage="stage1_control", strategy="naive", epochs=EPOCHS, seed=s, lr=lr, eval_split="test",
                           save_checkpoints=True, **arm)
               for s in SEEDS for lr, arm in [(0.1, dict(lr_schedule="flat")),
                                              (0.6, dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="fast_deep"))]]
cdf = table(OLD_CONTROL + CONTROL)
done = cdf[cdf["status"] != "not finished"] if "status" in cdf else cdf
assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
cols = ["avg_incremental_acc", "acc_just_learned", "final_avg_acc", "avg_forgetting", "minutes", "peak_gpu_mb"]
display(done.groupby(["arm", "lr"])[cols].agg(["mean", "std", "count"]).round(2))
lvl = [c for c in done.columns if c.startswith(("updates/", "rel_displacement/", "penalty_ratio/"))]
if lvl: display(done.groupby(["arm", "lr"])[lvl].mean().T.round(4))

avg_incremental_acc             acc_just_learned        \
                                    mean   std count             mean   std   
arm              lr                                                           
flat             0.1               14.08  0.52     3            82.37  0.14   
flat_nl          0.1               14.19  0.38     3            82.15  0.32   
llrd_fast_deep   0.6               14.12  0.36     3            82.24  0.54   
nested_b         0.6               14.17  0.49     3            81.93  0.77   
nested_d_lam0.01 0.7               14.01  0.39     3            83.12  0.41   

                           final_avg_acc             avg_forgetting        \
                     count          mean   std count           mean   std   
arm              lr                                                         
flat             0.1     3          4.35  0.18     3          82.13  0.18   
flat_nl          0.1     3          4.19  0.26     3          82.06  0.45   
llrd_fast_deep   0.6     3          4.33  0.24     3          82.00  0.82   
nested_b         0.6     3          4.21  0.37     3          81.81  1.18   
nested_d_lam0.01 0.7     3          4.29  0.19     3          82.98  0.33   

                           minutes             peak_gpu_mb              
                     count    mean   std count        mean   std count  
arm              lr                                                     
flat             0.1     3    8.87  0.49     3         NaN   NaN     0  
flat_nl          0.1     3    8.79  0.07     3      211.90  6.16     3  
llrd_fast_deep   0.6     3    9.10  0.54     3         NaN   NaN     0  
nested_b         0.6     3    8.65  0.06     3      215.09  3.08     3  
nested_d_lam0.01 0.7     3   19.98  0.08     3      220.43  3.08     3

arm,flat,flat_nl,llrd_fast_deep,nested_b,nested_d_lam0.01
lr,0.1,0.1,0.6,0.6,0.7
updates/stem+layer1,NaN,790.0000,NaN,790.0000,49.0000
rel_displacement/stem+layer1,NaN,0.0993,NaN,0.0577,0.0739
updates/layer2+layer3,NaN,790.0000,NaN,790.0000,197.0000
rel_displacement/layer2+layer3,NaN,0.0673,NaN,0.0821,0.0944
updates/layer4+fc,NaN,790.0000,NaN,790.0000,790.0000
rel_displacement/layer4+fc,NaN,0.0806,NaN,0.1704,0.1979
penalty_ratio/stem+layer1,NaN,NaN,NaN,NaN,0.4448
penalty_ratio/layer2+layer3,NaN,NaN,NaN,NaN,0.0762
